In [104]:
import pandas as pd
import nltk 
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy
from scipy.sparse import hstack
import numpy as np
import spacy
import re

In [105]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet = True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

True

In [106]:
# Load data
df = pd.read_csv('articles_clean.csv')

def truncate_to_25pct(text):
    if not isinstance(text, str): return ""
    sentences = nltk.sent_tokenize(text)
    limit = max(1, len(sentences) // 4)
    return " ".join(sentences[:limit])



In [107]:
# Process
df['full_text'] = df['full_text'].apply(truncate_to_25pct)

# Save
df.to_csv('articles_25%_sentences.csv', index=False)

# Load truncated data
df = pd.read_csv('articles_25%_sentences.csv')

df['full_text'] = df['full_text'].replace(r'\s+', ' ', regex=True).str.strip()

In [108]:
df['word_count'] = df['full_text'].str.split().str.len()
invalid_topics = df[df['word_count'] < 30]['topic_id'].unique()
df_clean = df[~df['topic_id'].isin(invalid_topics)]

print(f"Topics removed: {len(invalid_topics)}")
print(f"Residual topics: {df_clean['topic_id'].nunique()}")

Topics removed: 12
Residual topics: 57


In [109]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])

def mask_named_entities(text):
    doc = nlp(text)
    # Replace every named entity with its label (e.g., PERSON, GPE, ORG)
    for ent in reversed(doc.ents):
        text = text[:ent.start_char] + ent.label_ + text[ent.end_char:]
    return text

In [110]:
df['full_text'] = df['full_text'].apply(mask_named_entities)

In [111]:
def rigorous_clean(text):
    text = str(text)
    # Remove everything except alphanumeric, spaces, and basic punctuation
    text = re.sub(r'[^\w\s.,!?]', '', text)
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [112]:
df['full_text'] = df['full_text'].apply(rigorous_clean)

In [113]:
# POS Tagging extraction function
def get_pos_tags(text):
    tokens = nltk.word_tokenize(text)
    tags = nltk.pos_tag(tokens)
    return " ".join([tag for word, tag in tags])

In [114]:
# Extract POS tags
df['pos_text'] = df['full_text'].apply(get_pos_tags)

In [115]:
# Vectorize Char N-grams (2-4)
char_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), min_df=0.05, strip_accents='unicode')
char_features = char_vectorizer.fit_transform(df['full_text'].astype(str))

# Vectorize POS N-grams (1-2)
pos_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=0.05)
pos_features = pos_vectorizer.fit_transform(df['pos_text'])

X = hstack([char_features, pos_features])

print(f"Total number of stylometric features: {X.shape[1]}")
print(f"Total features: {X.shape[1]}")

Total number of stylometric features: 6521
Total features: 6521


In [116]:
# 1. Get feature names from both vectorizers
char_names = char_vectorizer.get_feature_names_out()
pos_names = pos_vectorizer.get_feature_names_out()
all_names = np.concatenate([char_names, pos_names])

# 2. Calculate mean TF-IDF score for each feature across the dataset
mean_scores = np.array(X.mean(axis=0)).flatten()

# 3. Create a DataFrame for visualization
feature_importance = pd.DataFrame({
    'feature': all_names,
    'importance': mean_scores
}).sort_values(by='importance', ascending=False)

# 4. Display results
print(feature_importance.head(20))

      feature  importance
6337      nnp    0.407180
6317       nn    0.377366
6283       in    0.324009
6269       dt    0.238028
6358      nns    0.163746
6296       jj    0.158097
1675       e     0.158013
533         t    0.126513
6442      vbd    0.115429
6320    nn in    0.114776
6345  nnp nnp    0.114346
6289   in nnp    0.112964
6273    dt nn    0.112659
5           a    0.106903
2938       in    0.105936
5081       s     0.100047
6284    in dt    0.098808
5622       th    0.097287
6379      prp    0.091258
6426       vb    0.086099
